## Scripted RAG

In [2]:
### 1.
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
### 2.
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS:
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, use:
```bash
curl http://localhost:11434
```

If you get a connection refused error while prompting, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [ ]:
# Scripted RAG fails because of typo - search fails to find relevant context
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about **Olama** specifically.

If you mean running the course locally, the FAQ says you can do that if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. Codespaces is just the easiest starting environment.

If you meant **Ollama**, please share the relevant FAQ context or wording, since it isn’t included here.


## Agentic RAG

In [ ]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

# Prompt with no context
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out by checking:\n- whether enrollment is still open\n- if there are prerequisites\n- whether there’s a waiting list\n- who to contact for approval\n\nIf you already have the course name, school/platform, or a link, send it and I’ll help you draft a quick message asking to join.'

In [4]:
### 3.
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
### 4.
# Serch tool schema

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
# Call with search tool provided
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment"}', call_id='call_hY1ToZlM0Gsih1omB5ixdTCd', name='search', type='function_call', id='fc_06754ef4d580bd08006a39538f585081a2bd66d6ac28062d10', namespace=None, status='completed')]

In [ ]:
import json

# Invoke search on the model's behalf
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [22]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [23]:
print(response.output)

[ResponseOutputMessage(id='msg_06754ef4d580bd08006a3954725cec81a28fc8f7751434ed28', content=[ResponseOutputText(annotations=[], text='Yes — you can still join and start learning. The course materials are available, and you can work through them at your own pace.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [ ]:
# Include conversation history
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment"}', call_id='call_hY1ToZlM0Gsih1omB5ixdTCd', name='search', type='function_call', id='fc_06754ef4d580bd08006a39538f585081a2bd66d6ac28062d10', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_hY1ToZlM0Gsih1omB5ixdTCd', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\

In [17]:
from pprint import pprint

pprint(messages)

[{'content': 'I just discovered the course. Can I join it?', 'role': 'user'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment"}', call_id='call_hY1ToZlM0Gsih1omB5ixdTCd', name='search', type='function_call', id='fc_06754ef4d580bd08006a39538f585081a2bd66d6ac28062d10', namespace=None, status='completed'),
 {'call_id': 'call_hY1ToZlM0Gsih1omB5ixdTCd',
  'output': '[\n'
            '  {\n'
            '    "id": "74eb249bbf",\n'
            '    "course": "llm-zoomcamp",\n'
            '    "section": "General Course-Related Questions",\n'
            '    "question": "I just discovered the course. Can I still '
            'join?",\n'
            '    "answer": "Yes, but if you want to receive a certificate, you '
            'need to submit your project while we\\u2019re still accepting '
            'submissions."\n'
            '  },\n'
            '  {\n'
            '    "id": "04919992b3",\n'
          

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning. The course materials are available, and you can work through them at your own pace.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [24]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(776, 51)

In [25]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


## Agentic Loop

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
import json

# Calls a tool (search function) and returns the dict describing that call (format expected by the model).
# This dict is later passed back to the model (conversation history).
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [19]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print("RESPONSE:")
print(response.output)

messages.extend(response.output)
has_function_calls = False

# Model can decide to make multiple function calls so we loop through its response.
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

RESPONSE:
[ResponseFunctionToolCall(arguments='{"query":"join the course can I still enroll discovered the course late FAQ"}', call_id='call_qxiyZjdpy67oO6lFRLY0v7Wk', name='search', type='function_call', id='fc_0c725fcb157fc365006a399d6afabc81918d948cd65e982821', namespace=None, status='completed')]
function_call: search {"query":"join the course can I still enroll discovered the course late FAQ"}


In [ ]:
# History (messages) content here:
# 1. Original instructions
# 2. Original query
# 3. Response from LLM with decision to call search function
# 4. Output from search function invocation (results from the index)

for msg in messages:
    print(msg)

{'role': 'developer', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}
{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
ResponseFunctionToolCall(arguments='{"query":"join the course late discovered the course can I join enrollment start anytime FAQ"}', call_id='call_5ZlzGa4J0tUG8Xrsv6krnwJc', name='search', type='function_call', id='fc_070ed01f6c5ce8e6006a399d0b622c819cb4c1e31e62243299', namespace=None, status='completed')
{'type': 'function_call_output', 'call_id': 'call_5ZlzGa4J0tUG8Xrsv6krnwJc', 'output': '[\n  {\n    "id": "74eb249bb

In [20]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can join the course even if you discovered it late. The materials are available, so you can start anytime.

One important caveat: if you want a certificate, you’ll need to submit your project while the course is still accepting submissions. Also, certificates are only available for the live cohort, not self-paced mode.

If you want, I can also help you figure out the best way to catch up quickly.


In [21]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [24]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment can I still join discovered the course"}
function_call: search {"query":"course FAQ registration add drop enroll late join"}
function_call: search {"query":"can I still join the course after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework while the forms are open, even if you didn’t register earlier.

If you want, I can also help you figure out the best way to catch up or explain the certificate requirements. Is there anything else you’d like to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework while the forms are open, even if you didn’t register earlier.\n\nIf you want, I can also help you figure out the best way to catch up or explain the certificate requirements. Is there anything else you’d like to explore?'

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama"}
function_call: search {"query":"Ollama run locally install models local machine"}
function_call: search {"query":"local Ollama setup command run"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Pick your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

  

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Pick your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error later, you can restart the server with:\n```bash\nollama serve\n```\n\nIf you\'d like, I can also show you how to use Ollama with a RAG app

In [30]:
# Encouraging the model to make multiple searches.
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple search calls. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enrollment late registration can I join course discovered the course"}
function_call: search {"query":"can I join the course if I just discovered it enrollment deadline waitlist registration"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework even if you didn’t register, as long as the homework form is open.

If you want, I can also tell you the best way to get started with the course materials.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework even if you didn’t register, as long as the homework form is open.\n\nIf you want, I can also tell you the best way to get started with the course materials.'

In [31]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. `d4 d5`
2. `c4`

White offers the c-pawn to try to **control the center** and draw Black’s `d5` pawn away from the center.

A few quick points:
- It’s one of the oldest and most famous chess openings.
- It’s called a “gambit” because White is offering a pawn.
- Black can accept or decline the pawn.
- Common main types include the **Queen’s Gambit Accepted** and **Queen’s Gambit Declined**.

If you want, I can also explain **why it’s strong** or show a **simple board example**.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. `d4 d5`\n2. `c4`\n\nWhite offers the c-pawn to try to **control the center** and draw Black’s `d5` pawn away from the center.\n\nA few quick points:\n- It’s one of the oldest and most famous chess openings.\n- It’s called a “gambit” because White is offering a pawn.\n- Black can accept or decline the pawn.\n- Common main types include the **Queen’s Gambit Accepted** and **Queen’s Gambit Declined**.\n\nIf you want, I can also explain **why it’s strong** or show a **simple board example**.'

In [15]:
# Restrict off-topic questions
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

# agent_loop(instructions, "what's queen gambit?")

## ToyAIKit

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [6]:
# Register the search tool
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [10]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [ ]:
# Let the ToyAIKit derive the tool schema from the function type hints and docstring
# Other AI frameworks can do the same
agent_tools = Tools()
agent_tools.add_tool(search)

In [12]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [13]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [16]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [18]:
# Runs the agentic loop. The typo is managed iteratively (first call with, a typo, then a second call with the correct spelling).
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [ ]:
# Output from OpenAIResponsesRunner invocation already has the token cost calculated.
result.cost

CostInfo(input_cost=Decimal('0.00271875'), output_cost=Decimal('0.0014355'), total_cost=Decimal('0.00415425'))

In [ ]:
# Conversation history
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local setup FAQ"}', call_id='call_SAdXD8KSbI

In [21]:
# Continue conversation - inlude message history from the previous calls

result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages, # History
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
# Interactive chat
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='Can I still join the course?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"join course still join late enrollment 